In [ ]:
import pandas as pd
import os
import json
import numpy as np
from os.path import dirname

root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print("CWD:", os.getcwd())
print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

In [ ]:
with open("dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [ ]:
list(datasets_info.keys())

In [ ]:
dataset = "bpic2012_O_DECLINED-COMPLETE" #Select dataset to execute on

In [ ]:
tab_all = pd.read_csv(f"datasets/processed/{dataset}_processed_all.csv")
tab_all.head()

In [ ]:
tab_train = pd.read_csv(f"datasets/processed/{dataset}_processed_train.csv")
tab_valid = pd.read_csv(f"datasets/processed/{dataset}_processed_valid.csv")
tab_test = pd.read_csv(f"datasets/processed/{dataset}_processed_test.csv")

In [ ]:
#if dataset.startswith("BPIC15"):
#    with open("dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)["BPIC15_common"]
#else:
#    with open("dataset_features.json", 'r') as file:
#        dataset_info = json.load(file)[dataset]


with open("dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]


In [ ]:
dataset_info

In [ ]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [ ]:
for k in categorical_columns:
    tab_all[k] = tab_all[k].astype("object")
    tab_train[k] = tab_train[k].astype("object")
    tab_valid[k] = tab_valid[k].astype("object")
    tab_test[k] = tab_test[k].astype("object")

for df in (tab_all, tab_train, tab_valid, tab_test):
        df["Label"] = (df["Label"]=="deviant").astype(int)

#cast Label to int (0/1) 
#if dataset == "BPI12_DECLINED_COMPLETE":
#    for df in (tab_all, tab_train, tab_valid, tab_test):
#        df["Label"] = df["Label"].astype(int)  
#elif dataset in ["BPIC11_f1"] or dataset.startswith("BPIC15"): #Remove old stuff
#    for df in (tab_all, tab_train, tab_valid, tab_test):
#        df["Label"] = (df["Label"]=="deviant").astype(int)
#else: 
#    raise ValueError(f"Unknown dataset name: {dataset!r}")

In [ ]:
#Check percentage true/false in test dataset to check it isn't blindly predicting a value.
unique_labels = tab_test.drop_duplicates('CaseID', keep='last')['Label']
percentage_true = unique_labels.mean() * 100

print(f"Test set: {percentage_true:.2f}% True")
print(f"Test set: {100-percentage_true:.2f}% False")

### Prepare the graphs

In [ ]:
import sklearn.preprocessing

from typing import List

In [ ]:
def get_case_ids(tab):
    return list(tab["CaseID"].unique())

In [ ]:
from torch import tensor, max, int64, float32
from torch_geometric.data import HeteroData

In [ ]:
def get_one_hot_encoder(dataset: pd.DataFrame, key: str):
    datas = np.unique(dataset[key].astype(str)).reshape(-1,1)
    onehot = sklearn.preprocessing.OneHotEncoder()
    onehot.fit(datas)
    return onehot

In [ ]:
def get_one_hot_encodings(onehot, datas: pd.Series):
    return onehot.transform(datas.reshape(-1, 1)).toarray()

In [ ]:
def get_node_features(dataset: pd.DataFrame, trace: pd.DataFrame, cat_features, real_features) -> dict:
 

    res = {}

    for key in trace:
        values = trace[key].values
        if key in cat_features:
            onehot_encoder = get_one_hot_encoder(dataset, key)
            try:
                res[key] = tensor(
                    get_one_hot_encodings(onehot_encoder, values),
                    dtype=float32,
                    requires_grad=True
                )
            except ValueError:
                print(key)
                print(values)
        if key in real_features:
            res[key] = tensor(values,  dtype=float32,requires_grad=True)
            res[key] = res[key].reshape(res[key].shape[0], 1)
        
    

    return res


In [ ]:
def compute_edges_indexs(node_features: dict, prefix_len):
    res = {}
    keys = node_features.keys()
    
    indexes = [[i, i + 1] for i in range(prefix_len-1)]
   
    for k in keys:
        if len(node_features[k]) != 1:
            if k == "Activity":
                res[(k, "followed_by", k)] = indexes
                for k2 in keys:
                    if k2 != k:
                        if len(node_features[k2]) == 1:
                            res[(k, "related_to", k2)] = [
                                [i, 0] for i in range(prefix_len)
                            ]
                        else:
                            res[(k, "related_to", k2)] = [
                                [i, i] for i in range(prefix_len)
                            ]
            else:
                res[(k, "related_to", k)] = indexes

    return res

In [ ]:

def build_prefixes_graph_from_trace(dataset, trace, cat_features, real_features, prefix_length):
    X = []  # graphs
   
    
    
    node_features = get_node_features(dataset, trace, cat_features, real_features)
    
    
    
    
    G = HeteroData()
        
        
        
    for k in node_features:
        if k != "Label":
            G[k].x = node_features[k][:prefix_length]


    edges_indexes = compute_edges_indexs(node_features, prefix_length)

    


    for k in edges_indexes:
        ce = [[], []]
        for i in range(len(edges_indexes[k])):
            ce[0].append(edges_indexes[k][i][0])
            ce[1].append(edges_indexes[k][i][1])
        edges_indexes[k] = ce

    for k in edges_indexes:
        G[k].edge_index = tensor(edges_indexes[k], dtype=int64)


    ## Get the label of the trace
    label_value = trace["Label"].iloc[0]
    G.y = tensor([int(label_value)], dtype=int64)
             
    X.append(G)
        
    return X

## Create the datasets

In [ ]:
case_train_ids = get_case_ids(tab_train)
case_valid_ids = get_case_ids(tab_valid)
case_test_ids = get_case_ids(tab_test)

In [ ]:
print(len(case_train_ids))
print(len(case_valid_ids))
print(len(case_test_ids))

In [ ]:
tab_train["CaseID"] = tab_train["CaseID"].astype(np.str_)
tab_valid["CaseID"] = tab_valid["CaseID"].astype(np.str_)
tab_test["CaseID"] = tab_test["CaseID"].astype(np.str_)

In [ ]:
trace = (
        tab_train.query(f"CaseID == '{case_train_ids[0]}'")
        .reset_index()
        .drop(columns="index")
        #.drop(columns="CaseID")
    )
trace 

In [ ]:
import pickle
from tqdm.notebook import tqdm

In [ ]:
min_len = tab_all.groupby("CaseID").size().min()
max_len = tab_all.groupby("CaseID").size().max()
print("Minimum trace length:", min_len)
print("Maximum trace length:", max_len)


In [ ]:
PREFIX_LENGTH = 4

In [ ]:
print("Preparing training dataset...")

X_train = []


for i in tqdm(range(len(case_train_ids))):
    trace = (
        tab_train.query(f"CaseID == '{case_train_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )

    if len(trace) > PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH,
        )
        for j in range(len(graphs)):
            X_train.append(graphs[j])

In [ ]:
print(dataset,"\n")

In [ ]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "wb") as f:
    pickle.dump(X_train, f)

In [ ]:
print("Preparing validation dataset...")

X_valid = []


for i in tqdm(range(len(case_valid_ids))):
    trace = (
        tab_valid.query(f"CaseID == '{case_valid_ids[i]}'")
        .reset_index()
        .drop(columns="index")
        .drop(columns="CaseID")
    )
    if len(trace) > PREFIX_LENGTH:
        graphs = build_prefixes_graph_from_trace(
            dataset=tab_all,
            trace=trace,
            cat_features=categorical_columns,
            real_features=real_value_columns,
            prefix_length=PREFIX_LENGTH
        )
        for i in range(len(graphs)):
            X_valid.append(graphs[i])

In [ ]:
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "wb") as f:
    pickle.dump(X_valid, f)

In [ ]:
# Find which label is the minority class
#last_labels = tab_test.drop_duplicates("CaseID", keep="last")[["CaseID","Label"]]
#pct_true   = last_labels["Label"].mean() * 100
#minority   = 0 if pct_true > 50 else 1

#Set of minority class CaseID
#minority_case_ids = (
#    last_labels[last_labels["Label"] == minority]["CaseID"].unique()
#)

#index CaseID, rows length of the trace for each CaseID 
#trace_lengths = tab_test.groupby("CaseID").size()
#we only care about the minority class ones.
#minority_lengths = trace_lengths.loc[minority_case_ids]

# 5) Find the smallest L (1…40) so that ≥ 90% of minority traces have length ≤ L:
#total_minority = len(minority_lengths)
#MaxPrefix = 40  # default if none smaller satisfies the condition

#for L in range(1, 41):
#    finished_count = (minority_lengths <= L).sum()
#    if finished_count >= 0.9 * total_minority:
#        MaxPrefix = L
#        break

#print(f"Minority = {minority}, total minority traces = {total_minority}")
#print(f"90th‐percentile threshold → MaxPrefix = {MaxPrefix}")

#Bench actually tells us it's 1 always for our datasets. hospital_billing_1 excl.
if dataset == "traffic_fines_1":
    MaxPrefix = 10
elif dataset == "BPIC17_O_Cancelled":
    minority = True
    minority_lengths = tab_test[tab_test["Label"]==minority].groupby("CaseID").size()
    total_minority = len(minority_lengths)
    q90 = int(np.ceil(minority_lengths.quantile(0.9)))
    MaxPrefix = min(20,q90)
    print(f"Minority = {minority}, total minority traces = {total_minority}")
    print(f"90th‐percentile threshold: {q90:.2f}  MaxPrefix: {MaxPrefix}")
else:
    minority = True
    minority_lengths = tab_test[tab_test["Label"]==minority].groupby("CaseID").size()
    total_minority = len(minority_lengths)
    q90 = int(np.ceil(minority_lengths.quantile(0.9)))
    MaxPrefix = min(40,q90)
    print(f"Minority = {minority}, total minority traces = {total_minority}")
    print(f"90th‐percentile threshold: {q90:.2f}  MaxPrefix: {MaxPrefix}")





In [ ]:
#dict stores one test-graph list per prefix length
X_tests={}

for L in range(1,MaxPrefix+1):
    print(f"Preparing test dataset {L}...")
    X_test_L = []
    
    for i in tqdm(range(len(case_test_ids))):
        trace = (
            tab_test.query(f"CaseID == '{case_test_ids[i]}'")
            .reset_index()
            .drop(columns="index")
            .drop(columns="CaseID")
        )
        
        if len(trace) >= L:  #generate_prefix_data in experiments/DatasetManager confirms >=
            graphs = build_prefixes_graph_from_trace(
                dataset=tab_all,
                trace=trace,
                cat_features=categorical_columns,
                real_features=real_value_columns,
                prefix_length=L
            )
            X_test_L.extend(graphs)
        
    X_tests[L] = X_test_L
        

In [ ]:
#with open(data_dir_graphs + dataset + "_TEST_repair.pkl", "wb") as f:
#    pickle.dump(X_test, f)



for L, X_test_L in X_tests.items():
    fname   = f"{dataset}_TEST{L}_repair.pkl"
    outpath = os.path.join(data_dir_graphs, fname)
    with open(outpath, "wb") as f:
        pickle.dump(X_test_L, f)
    print(f"Saved {len(X_test_L)} graphs for prefix {L} to {outpath}")
